# Day 39 — EDA (awaiting cleaned CSV)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
f=Path('../Customer_Churn_Prediction/data/processed/cleaned_customer_churn.csv')
assert f.exists(), 'Run Day 38 first'
df=pd.read_csv(f); df['ChurnFlag']=df.Churn.map({'Yes':1,'No':0}); assert df.ChurnFlag.notna().all()
out=Path('visualizations'); out.mkdir(exist_ok=True)
obs=[]
def save(name):
    plt.tight_layout(); plt.savefig(out/(name+'.png')); plt.show(); plt.close()
counts=df.Churn.value_counts(); counts.plot.bar(title='Overall churn distribution'); plt.ylabel('Customers'); save('overall')
obs.append(['Overall churn',f'{df.ChurnFlag.mean():.2%} of {len(df)} customers','Baseline for retention planning'])
if 'TenureMonths' in df: df['TenureGroup']=pd.cut(df.TenureMonths,[-1,12,36,np.inf],labels=['0–12','13–36','37+']) # first year, mid tenure, established
for c in ['TenureGroup','ContractType','PaymentMethod','Gender','InternetService','ComplaintRaised','TechSupport','SeniorCitizen']:
    if c not in df: continue
    g=df.groupby(c,observed=True).ChurnFlag.agg(['size','sum','mean']).sort_values('mean'); display(g)
    (g['mean']*100).plot.bar(title='Churn rate by '+c); plt.ylabel('Churn (%)'); plt.ylim(0,100); save(c)
    for label,row in g.iloc[[0,-1]].iterrows(): obs.append([f'{c}: {label}',f"{row['mean']:.2%}; n={int(row['size'])}",'Investigate group differences; small groups may be unstable; not causal'])
if 'MonthlyCharges' in df:
    df.boxplot(column='MonthlyCharges',by='Churn'); plt.suptitle(''); plt.title('Monthly charges by churn'); plt.ylabel('Monthly charges'); save('charges')
    g=df.groupby('Churn').MonthlyCharges.describe(); display(g); obs.append(['Monthly charges',g[['count','mean','50%','min','max']].to_string(),'Compare overlap and extreme values; no causal inference'])
for c in ['AvgMonthlyDataUsageGB','SupportCallsLast90Days','Age']:
    if c in df:
        g=df.groupby('Churn')[c].describe(); display(g); obs.append([c,g[['count','mean','50%']].to_string(),'Investigate observed differences'])
result=pd.DataFrame(obs,columns=['Observation','Evidence','Possible Business Meaning']); result.insert(0,'No.',range(1,len(result)+1)); display(result); result.to_csv('EDA_Observations.csv',index=False)
print('Review chart overlap, sample sizes and interpretations; select at least ten distinct supported observations.')
